# 📘 XGBoost Regression (Tutor-Style Notes)

## 1️⃣ Basic Idea

- XGBoost regression is a **boosting algorithm**  
- Builds trees **sequentially**  
- Each new tree learns from **residuals**  
- Best split is chosen using **Similarity Score (SS)** and **Gain**

---

## 2️⃣ Dataset (Small Example)

| Row | X | y  |
|-----|---|----|
| 1   | 1 | 5  |
| 2   | 2 | 7  |
| 3   | 3 | 9  |
| 4   | 4 | 11 |

---

## 3️⃣ Step 1: Initial Prediction

Initial prediction = **mean of target**

$$
\hat{y} = \frac{5+7+9+11}{4} = 8
$$

---

## 4️⃣ Step 2: Calculate Residuals

Residual formula:

$$
\text{Residual} = y - \hat{y}
$$

| Row | y  | $\hat{y}$ | Residual |
|------|----|----------|----------|
| 1    | 5  | 8        | -3       |
| 2    | 7  | 8        | -1       |
| 3    | 9  | 8        | +1       |
| 4    | 11 | 8        | +3       |

---

## 5️⃣ Step 3: Similarity Score (SS)

### Tutor Formula

$$
SS = \frac{(\text{sum of residuals})^2}{\text{number of residuals} + lr}
$$

Where:

- sum of residuals = sum of residuals in that node  
- number of residuals = number of data points in that node  
- lr = small regularization constant (often 0)

---

## 6️⃣ Step 4: Root Node SS

$$
\text{Sum of residuals} = -3 - 1 + 1 + 3 = 0
$$

$$
SS_{root} = \frac{0^2}{4} = 0
$$

---

## 7️⃣ Step 5: Try All Possible Splits

Possible splits:  
- $X \leq 1.5$  
- $X \leq 2.5$  
- $X \leq 3.5$

---

### Split 1: $X \leq 1.5$

Left: Row 1  
Right: Rows 2,3,4

**Left SS:**

$$
\frac{(-3)^2}{1} = 9
$$

**Right SS:**

$$
\frac{(-1 + 1 + 3)^2}{3} = \frac{3^2}{3} = 3
$$

---

### Gain

$$
Gain = SS_L + SS_R - SS_{parent}
$$

$$
Gain = 9 + 3 - 0 = 12
$$

---

### Split 2: $X \leq 2.5$

Left: Rows 1,2  
Right: Rows 3,4

**Left SS:**

$$
\frac{(-3 - 1)^2}{2} = \frac{16}{2} = 8
$$

**Right SS:**

$$
\frac{(1 + 3)^2}{2} = \frac{16}{2} = 8
$$

---

### Gain

$$
Gain = 8 + 8 - 0 = 16
$$

✅ **Best split**

---

### Split 3: $X \leq 3.5$

Left: Rows 1,2,3  
Right: Row 4

**Left SS:**

$$
\frac{(-3 - 1 + 1)^2}{3} = \frac{(-3)^2}{3} = 3
$$

**Right SS:**

$$
\frac{(3)^2}{1} = 9
$$

---

### Gain

$$
Gain = 3 + 9 - 0 = 12
$$

---

## 8️⃣ Step 6: Best Split Selection

| Split     | Gain  |
|-----------|--------|
| $X \leq 1.5$ | 12   |
| **$X \leq 2.5$** | **16 ✅** |
| $X \leq 3.5$ | 12   |

👉 **Choose $X \leq 2.5$**

---

## 9️⃣ Step 7: Leaf Output (Mean Residual)

Leaf value = mean residual

- Left Leaf:

$$
\frac{-3 - 1}{2} = -2
$$

- Right Leaf:

$$
\frac{1 + 3}{2} = 2
$$

---

## 🔟 Step 8: Update Predictions

$$
\text{New prediction} = \hat{y} + \eta \times (\text{leaf value})
$$

Assuming learning rate $\eta = 0.3$,

| X | Updated Prediction        |
|---|---------------------------|
| 1 | $8 - 0.6 = \mathbf{7.4}$  |
| 2 | $\mathbf{7.4}$            |
| 3 | $8 + 0.6 = \mathbf{8.6}$  |
| 4 | $\mathbf{8.6}$            |

---

## 1️⃣1️⃣ Key Points

- SS measures how similar residuals are  
- Gain helps select the best split  
- Learning rate applied after tree building  
- Works only for squared error loss

---

## 1️⃣2️⃣ One-line exam answer

> XGBoost regression selects the best split by maximizing gain calculated from similarity scores based on residuals.

---

### ✅ Final Note

This format will render perfectly in Jupyter notebook markdown cells with math beautifully displayed.



In [28]:
import pandas as pd
import numpy as np 
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error , r2_score

from xgboost import XGBRegressor

In [29]:
# Load dataset
diabetes = load_diabetes()

X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y = pd.Series(diabetes.target, name="target")

In [38]:
y

0      151.0
1       75.0
2      141.0
3      206.0
4      135.0
       ...  
437    178.0
438    104.0
439    132.0
440    220.0
441     57.0
Name: target, Length: 442, dtype: float64

In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42 
)

In [33]:
X_train

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
17,0.070769,0.050680,0.012117,0.056301,0.034206,0.049416,-0.039719,0.034309,0.027364,-0.001078
66,-0.009147,0.050680,-0.018062,-0.033213,-0.020832,0.012152,-0.072854,0.071210,0.000272,0.019633
137,0.005383,-0.044642,0.049840,0.097615,-0.015328,-0.016345,-0.006584,-0.002592,0.017036,-0.013504
245,-0.027310,-0.044642,-0.035307,-0.029770,-0.056607,-0.058620,0.030232,-0.039493,-0.049872,-0.129483
31,-0.023677,-0.044642,-0.065486,-0.081413,-0.038720,-0.053610,0.059685,-0.076395,-0.037129,-0.042499
...,...,...,...,...,...,...,...,...,...,...
106,-0.096328,-0.044642,-0.076264,-0.043542,-0.045599,-0.034821,0.008142,-0.039493,-0.059471,-0.083920
270,0.005383,0.050680,0.030440,0.083844,-0.037344,-0.047347,0.015505,-0.039493,0.008641,0.015491
348,0.030811,-0.044642,-0.020218,-0.005670,-0.004321,-0.029497,0.078093,-0.039493,-0.010903,-0.001078
435,-0.012780,-0.044642,-0.023451,-0.040099,-0.016704,0.004636,-0.017629,-0.002592,-0.038460,-0.038357


In [34]:
xgb = XGBRegressor()

In [35]:
xgb.fit(X_train , y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [36]:
y_pred = xgb.predict(X_test)

In [37]:
r2_score(y_test ,y_pred)

0.3675149756138415